# Predicting Mortgage Delinquency Risk

You have been hired by a mortgage servicing firm (a company that buys mortgages and then collects mortgage payments from homeowners) to build a model to answer the question: 

Given all available information about a newly issued mortgage, what is the likelihood that the mortgage will enter delinquency (the homeowner will be at least 30 days late on a mortgage payment) during the first two years of the mortgage?

The servicer's hope, obviously, is to differentiate between good mortgages they may wish to purchase (those that will be consistently paid) and mortgages they wish to avoid.

For this task, you have been given [REAL data on a sample of all US Standard single family home mortgages purchased or insured by Freddie Mac](https://www.freddiemac.com/research/datasets/sf-loanlevel-dataset) in a single calendar year along with payment data from that and two subsequent years.

### The Supervised Machine Learning Workflow

To answer this passive prediction question, we will use the most used machine learning package in Python: `scikit-learn`. In particular, we will use this library to try to predict which mortgages will enter delinquency using both logistic regression and a random forest model. 

The goal of this exercise is not to understand the nuances of these specific models, but instead to get a feel for the standard workflow used in Supervised Machine Learning (the approach we generally use to answer Passive Prediction questions). As you will see, the workflow for this kind of analysis is basically the same no matter what model you choose to use.

## Gradescope Autograding

Please follow [all standard guidance](https://www.practicaldatascience.org/ids720_specific/autograder_guidelines.html) for submitting this assignment to the Gradescope autograder, including storing your solutions in a dictionary called `results` and ensuring your notebook runs from the start to completion without any errors.

For this assignment, please name your file `exercise_passiveprediction_workflow.ipynb` before uploading.

You can check that you have answers for all questions in your `results` dictionary with this code:

```python
assert set(results.keys()) == {
    "ex1_target_var",
    "ex2_delinquency_rate",
    "ex3_num_features",
    "ex4_train_rows",
    "ex4_test_rows",
    "ex6_true_negatives",
    "ex5_baseline_accuracy",
    "ex5_baseline_f1",
    "ex8_rf_accuracy",
    "ex8_rf_f1",
    "ex9_better_model",
}
```

### Submission Limits

Please remember that you are **only allowed THREE submissions to the autograder.** Your last submission (if you submit 3 or fewer times), or your third submission (if you submit more than 3 times) will determine your grade. Submissions that error out will **not** count against this total.

### Exercise 1: Load and Explore the Data

Load the dataset from `balanced_cleaned_encoded.csv` which you can find [here](https://github.com/nickeubank/MIDS_Data/blob/master/mortgages/exercise_passiveprediction_workflow/balanced_cleaned_encoded.csv) and look at it. 

Given the goal of the assignment and the columns you find, store the name of the "target variable" that the mortgage servicing firm would want you to predict, and save it to `results["ex1_target_var"]`.

Some cleaning has been done to the dataset, but this is *real* mortgage data. You can also find the [original dataset here](https://github.com/nickeubank/MIDS_Data/tree/master/mortgages/2004). There is [supplemental documentation here](https://www.freddiemac.com/research/datasets/sf-loanlevel-dataset) on the dataset.


### Exercise 2: Check Class Balance

Inspect the distribution of `ever_delinquent` to understand how balanced the dataset is. A balanced dataset has roughly equal representation of each class (~50%) of the variable you are trying to predict.

Balance is an important attribute of data when doing supervised machine learning. When data is really imbalanced, models have a tendency to just predict that all observations have the value that appears most frequently in the data. 

To illustrate how this might work, suppose you had data on routine mammograms. Because routine mammograms are given to all women above a certain age as a precaution, the *vast* majority of routine mammograms are benign. Suppose you took mammogram data in which 99% of cases were benign and 1% were potentially malignant (might indicate breast cancer), and asked a model to predict whether a given mammogram was benign or malignant. If the model just said "this is benign" for **all** mammograms, it would be correct 99% of the time, and thus have a 99% accuracy rate. 

In real-world mortgage data, delinquencies are rare, as most loans are repaid on time, so the raw data is destined to be imbalanced. To help prevent the imbalance problem describe above, we've sampled the data to improve the balance — one of several strategies to address imbalanced data. 

Store the overall delinquency rate (share of rows where `ever_delinquent == 1`) **rounded to 4 decimal places** in `results["ex2_delinquency_rate"]`.

### Exercise 3: Define X and y

When working with `statsmodels`, we were able to fit our regressions by passing a DataFrame and a formula string:

```python
# Fit model
import statsmodels.formula.api as smf

my_model = smf.ols(
    "income ~ age + C(education) + female",
    data=salaries,
).fit()
```

In genera, however, most machine learning algorithms (like those in scikit-learn) don't directly support formulas. Instead, for a dataset with `N` observations and `M` explanatory variables, they expect to be given to two arrays: a one-dimensional vector of length `N` containing the values of the variable you want to learn to predict, and an `NxM` "feature matrix" with the explanatory variables.

Separate the feature matrix `X` from the target vector `y`.

Store the number of feature columns (`M`) in `results["ex3_num_features"]`.

### Exercise 4: Train/Test Split

If you were trained in statistical inference by economists or social scientists, you were likely taught that you should use theory and domain expertise to decide on what variables go into your model, and whether to use interaction terms, log the variables, etc. Theory, I know I was taught as a graduate student, is *the* thing that determines whether our model is correctly specified.

Supervised Machine Learning takes a different approach to modelling. Because Supervised Machine Learning often uses *extremely* flexible models, it will often be the case that the model will *overfit* the data. *Overfitting* is when a model tries to accommodate all of the individual observations in the dataset being used for training in a way that causes it to not be very good when it sees new data.

Consider the following illustration. The plot shows a set of points (blue) generated with the model $y =  18x + \epsilon$ where $\epsilon \sim Normal(0, 5)$. The true relationship between X and Y is given by the green line. The red line is an example of a model that is massively *overfit* — it has bent itself every which way in order to perfectly fit the specific observations in the data rather than try to model the underlying distribution. As a result, the model has zero errors *for these specific observation data*. But if we asked this model about values for observations that weren't in the training data — like $x=0.5$, the answer it gives us would be pretty useless. 

![overfitting](https://upload.wikimedia.org/wikipedia/commons/9/96/Pyplot_overfitting.png)

To help address this problem, in Supervised Machine Learning we usually start by splitting our data into a test dataset (often about 20% of the data) and a training dataset (80%). We then fit our data on the training dataset, then evaluate it by seeing how it performs at predicting values in the test dataset (which it didn't see during training).

You *can* do this just by sampling the data, but `scikit-learn` offers a nice convenience function that is commonly used.



Use `train_test_split` from `sklearn.model_selection` to split the data into a training set and a test set. Note that `train_test_split` will give you back four objects: `X_train, X_test, y_train, y_test`. Thus a common syntax for using it is something like:

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)
```

- Use 80% for training and the rest for testing (`test_size=0.2`).
- Use the `random_state=42` argument so results are reproducible.

Store the number of rows in the training set in `results["ex4_train_rows"]` and the number of rows in the testing set in `results["ex4_test_rows"]`. 

### Exercise 5: Fit Baseline Model (Logistic Regression)

Now that we've set aside a `test` dataset we can use later to evaluate our model, let's use a logistic regression to predict our target variable (delinquency) using our features!

Fit a `LogisticRegression` model as the baseline classifier. You can read more about logistic regression and how to import it on the Scikit-learn [documentation](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression).

Set the solver to `liblinear` and the max iterations to `1000`. Specify the `random_state=42` for reproducibility.

Basically, you will first create a model object with `LogisticRegression`, then fit it to your data using the `.fit()` method and your training data.

**Note:** with `statsmodels`, when you use `.fit` you get back a new object. In scikit-learn, `.fit` modifies the model object in place. So don't do:

```python
my_model = my_model.fit(X_train, y_train)
``` 

```python
my_model.fit(X_train, y_train)
```

### Exercise 6: Evaluate Model

To see how our model has performed, let's let it predict the target variable on our test data. We do this by passing our `X_test` data to the `.predict()` method of our fit model. Then we can compare the predicted values of the target variable to the true values. 

```python
y_pred_lr = lr.predict(X_test)
```

The most intuitive way to compare predictions to true values when dealing with discrete target variables (like `ever_delinquent`, which is only ever 0 or 1) is with a "confusion matrix", which you can easily do with the `confusion_matrix` and `ConfusionMatrixDisplay` functions in scikit-learn. In the case, a confusion matrix is a 2x2 matrix with the true value of our target variable on one side and the values predicted by the model on the other. Each cell then has the number of observations in each category. The goal, obviously, is to have as many observations as possible along the diagonal where the predicted value is equal to the true value.

Use code along these lines to make a confusion matrix:

```python
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm_lr = confusion_matrix(y_test, y_pred_lr)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_lr, display_labels=["Not Delinquent", "Delinquent"]
)
disp.plot(cmap="Blues")
```

Using the confusion_matrix object, get the number of observations in your data that were actually Not Delinquent that were also predicted to be Not Delinquent (i.e., the number of True Negatives) and store it in `results["ex6_true_negatives"]`.

**Note** The terms "positive" and "negative" are arbitrary — since `ever_delinquent` is 1 for a delinquent mortgage and 0 for a non-delinquent mortgage, Python will think about a delinquent mortgage as a "positive" observation and a non-delinquent as a "negative" observation. But we could have flipped them if we'd wanted to.

### Exercise 7: Scores

There are a near infinite number of ways to evaluate a model like this. Indeed, [if you hop over to wikipedia](https://en.wikipedia.org/wiki/Confusion_matrix#Table_of_confusion), you can see the *many* ways that you can measure performance just by adding up different elements of the confusion matrix and dividing by different elements. For example, accuracy is the sum of true positives and true negatives divided by total number of observations; sensitivity is the number of True Positives (delinquent mortgages predicted as delinquent) divided by the total number of positive observations), etc.

While you *can* calculate all these by hand, scikit-learn also provides some convenience functions in the [sklearn.metrics module](https://scikit-learn.org/stable/api/sklearn.metrics.html).

To illustrate, import and estimate both the model's `accuracy_score` and `f1_score` (you can lookup F1 in that wikipedia page). Store the test accuracy in `results["ex5_baseline_accuracy"]` and the F1 score in `results["ex5_baseline_f1"]`. **Round both to 4 decimal places.**




After fitting, generate predictions using the `predict` method and your test data. Generate:

- Accuracy, Precision, Recall, and F1 Score

Store the test accuracy in `results["ex5_baseline_accuracy"]` and the F1 score in `results["ex5_baseline_f1"]`. **Round both to 4 decimal places.**

## Machine Learning Workflow Summary

Congratulations! You just did fit your first machine learning algorithm! And you also learned that sometimes what constitutes "supervised machine learning" is more about your objectives and how you evaluate your model given that fitting a logistic regression is the same thing you've likely done in a standard statistics class.

But hopefully that's given you a general sense for the work-flow of scikit-learn:

1. **Split** your data:

```python 
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, y, test_size=0.2, train_size=0.8, random_state=42
)
```

2. **Fit** a model:

```python
from sklearn.linear_model import LinearRegression

my_model = LinearRegression()
my_model.fit(X_train, y_train)
```

3. **Predict** target variable values for the test data.

```python
my_predictions = my_model.predict(X_test)
```

4. **Evaluate** your predictions:

```python
my_model.score(X_test, y_test)
```

Yup, **split, fit, predict**: if you understand that work flow, you're well on your way to understanding Supervised Machine Learning.

### Exercise 8: Second Model (Random Forest Classifier)

One of the great advantages of the `scikit-learn` API is that models are interchangeable. You can swap in a completely different algorithm with minimal code changes.

To illustrate, let's fit a `RandomForestClassifier` using the exact same train/test split from previous exercise. Random Forests are an ensemble of decision trees that tend to capture non-linear relationships and interaction effects that Logistic Regression cannot. You can find it in `sklearn.ensemble`. Use `n_estimators=80`, `max_depth=2` and `random_state=42` when initializing the model.

Just trade `RandomForestClassifier` in for `LogisticRegression` and use the same `.fit()`, `.predict()`, `accuracy_score`, `f1_score`, and `confusion_matrx` code you already have. scikit-learn is plug-and-play.

Evaluate this model with the same metrics as the baseline model. Store test accuracy in `results["ex8_rf_accuracy"]` and F1 score in `results["ex8_rf_f1"]`. **Round both to 4 decimal places.**

### Exercise 9: Plot the Random Forest Confusion Matrix

Plot the confusion matrix for your Random Forest predictions against the true labels.  

### Exercise 10: Compare Models and Conclude

- Which model performed better overall?
- Given the business context (avoiding costly delinquent loans), which metric do you think matters most here? Why?

Store the name of the better model (by F1 score) as a string in `results["ex9_better_model"]`. Use either `"logistic_regression"` or `"random_forest"`.

## Is That It?

So is that all there is to this magical machine learning?

No, not exactly. Understanding the split-fit-predict workflow, and how easy it is to swap out models once it's setup, is a _big_ part of supervised machine learning. But we've glossed over a couple things:

- **How do you decide what "score" you want to maximize?** This, in my view, is the least appreciated but most important part of machine learning, because it's not a *technical* question, it's a *substantive* question. What errors do you care about most? How do you want to balance different goals for the model? Answering those questions means talking to stakeholders and thinking about how the model will be used.
- **Hyper-parameter tuning**: You know those values we told you to give to the models (`n_estimators=80`, `max_depth=2`)? Those are called "hyper-parameters," parameters that the model won't just pick on its own when you call `.fit()`. Picking those correctly is also important, though it isn't as hard as deciding on the right score to maximize. Often, you just loop over a lot of values, fit the model, and see how it does. 
- **Feature engineering**: we didn't think a lot about what variables we gave to the model. Feature engineering is what machine learning people call "picking your explanatory variables", and it is a bit of an art. 

Some of this we'll get into shortly, some is beyond the scope of this course, but none of it should feel out of reach — most machine learning is about experimenting and seeing what happens, and now that you know how to implement models, you're ready to do just that!